# FluxPipeline - Seed Management

Master seed control for reproducible and varied image generation.

## What You'll Learn
- Understand seed profiles
- Create reproducible results
- Control variation levels
- Find and save good seeds

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import torch
import matplotlib.pyplot as plt
import json

from pipeline import FluxPipeline
from core import SeedProfile, SeedManager
from config import setup_environment, logger
from utils import setup_workspace

setup_environment()
workspace = setup_workspace()
pipeline = FluxPipeline(workspace=workspace)
pipeline.load_model()
print("✅ Ready!")

## Understanding Seed Profiles

FluxPipeline provides 3 seed profiles:
- **CONSERVATIVE**: Low variation (42-9999)
- **BALANCED**: Medium variation (10000-999999) - default
- **CREATIVE**: High variation (1000000-2147483647)

In [ ]:
# Generate same prompt with different profiles
test_prompt = "A mystical forest with glowing mushrooms"

profiles = [
    SeedProfile.CONSERVATIVE,
    SeedProfile.BALANCED,
    SeedProfile.CREATIVE
]

profile_results = []

for profile in profiles:
    print(f"\nGenerating with {profile.name} profile...")
    
    # Generate 3 variations
    variations = []
    for i in range(3):
        image, seed = pipeline.generate_image(
            prompt=test_prompt,
            seed_profile=profile,
            height=512,
            width=512,
            num_inference_steps=4
        )
        if image:
            variations.append((image, seed))
            print(f"  Variation {i+1}: seed {seed}")
    
    profile_results.append((profile.name, variations))

# Display all variations
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, (profile_name, variations) in enumerate(profile_results):
    for col, (image, seed) in enumerate(variations):
        axes[row, col].imshow(image)
        axes[row, col].axis('off')
        axes[row, col].set_title(f"{profile_name}\nSeed: {seed}", fontsize=9)

plt.suptitle(test_prompt, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Reproducible Generation

Use a fixed seed to generate the same image every time.

In [ ]:
# Generate the same image multiple times
fixed_seed = 42
prompt = "A serene Japanese garden with cherry blossoms"

print(f"Generating 4 images with seed {fixed_seed}...\n")

reproducible_images = []

for i in range(4):
    image, seed = pipeline.generate_image(
        prompt=prompt,
        seed=fixed_seed,  # Fixed seed
        height=512,
        width=512,
        num_inference_steps=4
    )
    if image:
        reproducible_images.append(image)
        print(f"  Generation {i+1}: seed {seed}")

# Verify they're identical
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, image in enumerate(reproducible_images):
    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(f"Generation {idx+1}", fontsize=12)

plt.suptitle(f"All generated with seed {fixed_seed} (should be identical)", fontsize=14)
plt.tight_layout()
plt.show()

print("\n✅ All images should be identical!")

## Finding Good Seeds

Generate multiple variations and save the best seeds.

In [ ]:
# Generate multiple variations
prompt = "A majestic dragon in flight over mountains"
num_variations = 9

print(f"Generating {num_variations} variations to find good seeds...\n")

variations = []
seeds_used = []

for i in range(num_variations):
    image, seed = pipeline.generate_image(
        prompt=prompt,
        seed_profile=SeedProfile.BALANCED,
        height=512,
        width=512,
        num_inference_steps=4
    )
    if image:
        variations.append((image, seed))
        seeds_used.append(seed)
        print(f"  Variation {i+1}: seed {seed}")

# Display all variations
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

for idx, (image, seed) in enumerate(variations):
    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(f"Seed: {seed}", fontsize=10)

plt.suptitle(prompt, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nSeeds used: {seeds_used}")
print("\n💡 Note which seeds produced the best results!")

## Save Your Favorite Seeds

In [ ]:
# Create a seed library
seed_library = {
    "landscapes": {
        "description": "Seeds that work well for landscapes",
        "seeds": [42, 7777, 123456, 888888]
    },
    "portraits": {
        "description": "Seeds that work well for portraits",
        "seeds": [1234, 5678, 91011]
    },
    "fantasy": {
        "description": "Seeds for fantasy/magical scenes",
        "seeds": [9999, 777777, 314159]
    }
}

# Save to file
seed_library_path = workspace / "seed_library.json"
with open(seed_library_path, 'w') as f:
    json.dump(seed_library, f, indent=2)

print(f"✅ Seed library saved to: {seed_library_path}")
print("\nLibrary contents:")
for category, data in seed_library.items():
    print(f"\n{category.upper()}:")
    print(f"  {data['description']}")
    print(f"  Seeds: {data['seeds']}")

## Using Saved Seeds

In [ ]:
# Load and use seeds from library
with open(seed_library_path, 'r') as f:
    loaded_library = json.load(f)

# Use a landscape seed
landscape_seeds = loaded_library["landscapes"]["seeds"]
chosen_seed = landscape_seeds[0]  # Use first seed

print(f"Using saved landscape seed: {chosen_seed}\n")

landscape_prompt = "A peaceful mountain valley at sunrise"
image, seed = pipeline.generate_image(
    prompt=landscape_prompt,
    seed=chosen_seed,
    height=768,
    width=768,
    num_inference_steps=4
)

if image:
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"{landscape_prompt}\nSeed: {seed}", fontsize=12)
    plt.show()

## Seed Exploration

Explore variations around a good seed.

In [ ]:
def explore_nearby_seeds(base_seed, prompt, range_size=100, num_samples=6):
    """Generate images with seeds near a base seed."""
    print(f"Exploring seeds near {base_seed}...\n")
    
    results = []
    
    # Generate with nearby seeds
    for i in range(num_samples):
        # Vary seed within range
        seed = base_seed + (i * range_size // num_samples)
        
        image, actual_seed = pipeline.generate_image(
            prompt=prompt,
            seed=seed,
            height=512,
            width=512,
            num_inference_steps=4
        )
        
        if image:
            results.append((image, actual_seed))
            print(f"  Generated with seed {actual_seed}")
    
    return results

# Explore around seed 42
base_seed = 42
exploration_prompt = "A cozy library with warm lighting"

nearby_results = explore_nearby_seeds(
    base_seed,
    exploration_prompt,
    range_size=500,
    num_samples=6
)

# Display exploration results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (image, seed) in enumerate(nearby_results):
    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(f"Seed: {seed}", fontsize=10)

plt.suptitle(f"Seeds near {base_seed}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Seed Best Practices

### When to Use Fixed Seeds:
- You found a result you love and want to recreate it
- Testing prompt variations with consistent base
- Creating matched sets of images
- Debugging and troubleshooting

### When to Use Random Seeds:
- Exploring creative possibilities
- Finding the perfect composition
- Batch generation for variety
- Initial experimentation

### Seed Profile Guidelines:
- **CONSERVATIVE**: Best for subtle variations and consistency
- **BALANCED**: Good default for most use cases
- **CREATIVE**: Maximum variety and exploration

## Seed Statistics

In [ ]:
# Analyze seed distribution
import random

def get_seed_samples(profile, num_samples=100):
    """Get sample seeds from a profile."""
    seeds = []
    for _ in range(num_samples):
        if profile == SeedProfile.CONSERVATIVE:
            seeds.append(random.randint(42, 9999))
        elif profile == SeedProfile.BALANCED:
            seeds.append(random.randint(10000, 999999))
        elif profile == SeedProfile.CREATIVE:
            seeds.append(random.randint(1000000, 2147483647))
    return seeds

# Visualize seed distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, profile in enumerate([SeedProfile.CONSERVATIVE, SeedProfile.BALANCED, SeedProfile.CREATIVE]):
    samples = get_seed_samples(profile, 1000)
    
    axes[idx].hist(samples, bins=50, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f"{profile.name}\nRange: {min(samples):,} - {max(samples):,}", fontsize=12)
    axes[idx].set_xlabel('Seed Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Seed Distribution by Profile', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusion

You now know how to:
- ✅ Use different seed profiles for varied results
- ✅ Create reproducible images with fixed seeds
- ✅ Build and use a seed library
- ✅ Explore variations around good seeds

### Keep Experimenting!
The best way to find great seeds is to generate lots of images and keep track of which seeds produce the results you like.

In [ ]:
# Cleanup
import gc
del pipeline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Done! Happy generating!")